# Notebook 01 — Análisis Exploratorio de Datos (EDA)
**Dataset: Weather in Szeged 2006–2016 · Inteligencia Computacional · USB Medellín**

## 1. Cargar librerías y dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Ruta del dataset (ajustar si es necesario)
RUTA = "data/raw/weatherHistory.csv"

df = pd.read_csv(RUTA)
print("Shape:", df.shape)
df.head()

## 2. Información general del dataset

In [ ]:
print("Columnas y tipos de dato:")
print(df.dtypes)
print()
print("Valores nulos por columna:")
print(df.isnull().sum())
print()
print("Registros con Pressure = 0 (error sensor):", (df["Pressure (millibars)"] == 0).sum())

## 3. Estadísticas descriptivas

In [ ]:
cols_num = ["Temperature (C)", "Apparent Temperature (C)",
            "Humidity", "Wind Speed (km/h)", "Pressure (millibars)"]
df[cols_num].describe().round(3)

## 4. Distribuciones de las variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("Distribución de variables · Dataset Szeged", fontsize=13)

variables = ["Temperature (C)", "Apparent Temperature (C)",
             "Humidity", "Wind Speed (km/h)", "Pressure (millibars)"]
colores = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"]

for ax, var, color in zip(axes.flat, variables, colores):
    ax.hist(df[var], bins=50, color=color, alpha=0.75, edgecolor="white")
    ax.set_title(var, fontsize=9)
    ax.set_ylabel("Frecuencia")
    ax.grid(True, alpha=0.3)

axes[1, 2].set_visible(False)
plt.tight_layout()
plt.savefig("results/figures/fig1_distribuciones.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardada: fig1_distribuciones.png")

## 5. Matriz de correlaciones

In [ ]:
corr = df[cols_num].corr()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im)
etiquetas = ["Temp", "Temp.Ap", "Humedad", "Viento", "Presion"]
ax.set_xticks(range(len(cols_num)))
ax.set_yticks(range(len(cols_num)))
ax.set_xticklabels(etiquetas, rotation=30)
ax.set_yticklabels(etiquetas)
for i in range(len(cols_num)):
    for j in range(len(cols_num)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Matriz de correlaciones")
plt.tight_layout()
plt.savefig("results/figures/fig2_correlaciones.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardada: fig2_correlaciones.png")

## 6. Limpieza y transformaciones

In [ ]:
# Eliminar columnas que no se usan
df_limpio = df.drop(columns=["Formatted Date", "Summary", "Daily Summary",
                              "Loud Cover", "Wind Bearing (degrees)", "Precip Type"])

# Corregir Pressure = 0
mediana_p = df_limpio[df_limpio["Pressure (millibars)"] > 0]["Pressure (millibars)"].median()
df_limpio["Pressure (millibars)"] = df_limpio["Pressure (millibars)"].replace(0, mediana_p)

# Raíz cuadrada a Wind Speed
df_limpio["Wind Speed (km/h)"] = np.sqrt(df_limpio["Wind Speed (km/h)"])

# Renombrar
df_limpio = df_limpio.rename(columns={
    "Temperature (C)": "temp",
    "Apparent Temperature (C)": "temp_aparente",
    "Humidity": "humedad",
    "Wind Speed (km/h)": "viento",
    "Visibility (km)": "visibilidad",
    "Pressure (millibars)": "presion"
})

import os
os.makedirs("data/processed", exist_ok=True)
df_limpio.to_csv("data/processed/weather_clean.csv", index=False)
print("Dataset limpio guardado en data/processed/weather_clean.csv")
print("Shape:", df_limpio.shape)
df_limpio.head()

## 7. Variación temporal de la temperatura aparente

In [ ]:
# Temperatura aparente a lo largo del tiempo (muestra de 2000 registros)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_limpio["temp_aparente"].values[:2000], color="#e74c3c", linewidth=0.8, alpha=0.8)
ax.set_title("Temperatura aparente · Primeros 2000 registros horarios")
ax.set_xlabel("Hora")
ax.set_ylabel("°C")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/fig3_serie_temporal.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardada: fig3_serie_temporal.png")